# AI Writing Check — hosted from Colab

Run the one cell below (**Runtime → Run all**). It installs the app on this Colab machine,
starts it, and opens a public **Cloudflare tunnel** so you can use it from any browser.
No Cloudflare account or token is needed.

- First launch takes about 2 minutes (it downloads a ~600 MB model).
- A GPU runtime is optional. On CPU a 500-word document takes a couple of seconds.
- The link works only while this cell is running. Stop the cell to shut everything down.
- **Anyone who has the link can use the app**, and documents they submit are processed on
  this Colab machine. Share it accordingly.


In [ ]:
#@title Launch AI Writing Check { display-mode: "form" }
PORT = 8000  #@param {type:"integer"}
KEEP_RUNNING = True  #@param {type:"boolean"}

import os, re, subprocess, sys, time, urllib.request

REPO = "https://github.com/KNN-07/ai-writing-check.git"
APP = "/content/ai-writing-check"
CLOUDFLARED = "/content/cloudflared"
LOGS = {"server": "/content/server.log", "tunnel": "/content/tunnel.log"}

def run(*command):
    subprocess.run(command, check=True)

print("1/4  Getting the app…")
if os.path.isdir(APP):
    run("git", "-C", APP, "pull", "-q", "--ff-only")
else:
    run("git", "clone", "-q", "--depth", "1", REPO, APP)
run(sys.executable, "-m", "pip", "install", "-q",
    "fastapi", "uvicorn[standard]", "python-multipart", "pypdf", "python-docx")
if not os.path.exists(CLOUDFLARED):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED)
    os.chmod(CLOUDFLARED, 0o755)

print("2/4  Downloading the detection model (first run only)…")
sys.path.insert(0, APP)
from aicheck.detector import DEFAULT_MODEL
from huggingface_hub import snapshot_download
snapshot_download(DEFAULT_MODEL, allow_patterns=["*.json", "*.safetensors", "*.txt"])

print("3/4  Starting the app…")
subprocess.run(["pkill", "-f", "aicheck.server:app"])   # stop copies left by an earlier run
subprocess.run(["pkill", "-f", CLOUDFLARED])
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "aicheck.server:app", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd=APP, stdout=open(LOGS["server"], "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2)
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("The app failed to start:\n" + open(LOGS["server"]).read()[-2000:])
        time.sleep(1)
else:
    raise RuntimeError("The app did not come up:\n" + open(LOGS["server"]).read()[-2000:])

print("4/4  Opening the Cloudflare tunnel…")
tunnel = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=open(LOGS["tunnel"], "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(90):
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open(LOGS["tunnel"]).read())
    if found:
        url = found.group()
        break
    time.sleep(1)
if not url:
    raise RuntimeError("No tunnel address appeared:\n" + open(LOGS["tunnel"]).read()[-2000:])

print("\n" + "=" * 64)
print("  AI Writing Check is live at:\n\n  " + url)
print("\n  The address can take up to a minute to start responding.")
print("  Keep this cell running; stop it to shut the app down.")
print("=" * 64)

if KEEP_RUNNING:
    try:
        while server.poll() is None and tunnel.poll() is None:
            time.sleep(5)
        print("Stopped unexpectedly. See", LOGS)
    except KeyboardInterrupt:
        print("Shutting down.")
    finally:
        server.terminate()
        tunnel.terminate()
